In [2]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

from scipy import stats

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

from google.colab import data_table
data_table.enable_dataframe_formatter()



```
config = {
    "embedding_dim":32,
    "hidden_dim":64,
    "lr":1e-4,
    "batch_size":128,
    "epochs":30
}
```



In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Importing pandas dataframe and aligning it according to the problem.

In [5]:
columns = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL'] + [f'genre_{i}' for i in range(19)]

movie_details = pd.read_csv(
    "/content/drive/MyDrive/ml-100k/u.item",
    sep='|',
    encoding='latin-1',
    names=columns,
    header=None
)
movie_details = movie_details.drop(columns = 'video_release_date')
movie_details['movie_id'] -= 1

# Printing head of Movie data
print(movie_details.head(), end = '\n----------------------------------\n')

print(f"The shape of the movie details dataframe is {movie_details.shape}")

   movie_id        movie_title release_date                                           IMDb_URL  genre_0  genre_1  genre_2  genre_3  genre_4  genre_5  genre_6  genre_7  genre_8  genre_9  genre_10  genre_11  genre_12  genre_13  genre_14  genre_15  genre_16  genre_17  genre_18
0         0   Toy Story (1995)  01-Jan-1995  http://us.imdb.com/M/title-exact?Toy%20Story%2...        0        0        0        1        1        1        0        0        0        0         0         0         0         0         0         0         0         0         0
1         1   GoldenEye (1995)  01-Jan-1995  http://us.imdb.com/M/title-exact?GoldenEye%20(...        0        1        1        0        0        0        0        0        0        0         0         0         0         0         0         0         1         0         0
2         2  Four Rooms (1995)  01-Jan-1995  http://us.imdb.com/M/title-exact?Four%20Rooms%...        0        0        0        0        0        0        0        0        0

In [6]:

user_details = pd.read_csv(
    filepath_or_buffer = '/content/drive/MyDrive/ml-100k/u.user',
    sep = '|',
    encoding = 'latin-1',
    names = ['userid', 'age', 'gender', 'occupation', 'zip code'],
    header = None
)

print(f"The head of user data is: \n{user_details.head()}", end = '\n------------------\n')

user_details['gender'] = user_details['gender'].map({'M' : 0, 'F' : 1}).astype(np.int64)

user_details['userid'] -= 1

le = LabelEncoder()
user_details['occupation'] = le.fit_transform(user_details['occupation'])

age_mean = user_details['age'].mean()
age_std = user_details['age'].std()
user_details['age'] = (user_details['age'] - age_mean) / age_std

print(f"The head of user data is: \n{user_details.head()}", end = '\n------------------\n')

print(f"The information of user details dataframe :{user_details.info()}")

The head of user data is: 
   userid  age gender  occupation zip code
0       1   24      M  technician    85711
1       2   53      F       other    94043
2       3   23      M      writer    32067
3       4   24      M  technician    43537
4       5   33      F       other    15213
------------------
The head of user data is: 
   userid       age  gender  occupation zip code
0       0 -0.824422       0          19    85711
1       1  1.554043       1          13    94043
2       2 -0.906438       0          20    32067
3       3 -0.824422       0          19    43537
4       4 -0.086278       1          13    15213
------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 943 entries, 0 to 942
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   userid      943 non-null    int64  
 1   age         943 non-null    float64
 2   gender      943 non-null    int64  
 3   occupation  943 non-null    int64  
 4   z

In [7]:
data = pd.read_csv(
    filepath_or_buffer='/content/drive/MyDrive/ml-100k/u.data',
    sep = '\t',
    names = ['user_id', 'movie_id', 'rating', 'timestamp']
)

data = data.drop('timestamp', axis = 1)

data.shape

data.iloc[5]

data['user_id'] -= 1
data['movie_id'] -= 1

print(data.nunique())
data.head()

user_id      943
movie_id    1682
rating         5
dtype: int64


,user_id,movie_id,rating
0,195,241,3
1,185,301,3
2,21,376,1
3,243,50,2
4,165,345,1


In [8]:
print(data.shape, user_details.shape, movie_details.shape)

print(data.columns, user_details.columns, movie_details.columns)

(100000, 3) (943, 5) (1682, 23)
Index(['user_id', 'movie_id', 'rating'], dtype='object') Index(['userid', 'age', 'gender', 'occupation', 'zip code'], dtype='object') Index(['movie_id', 'movie_title', 'release_date', 'IMDb_URL', 'genre_0', 'genre_1', 'genre_2', 'genre_3', 'genre_4', 'genre_5', 'genre_6', 'genre_7', 'genre_8', 'genre_9', 'genre_10', 'genre_11', 'genre_12', 'genre_13', 'genre_14', 'genre_15', 'genre_16', 'genre_17', 'genre_18'], dtype='object')


# Converting raw dataframes to tenosrs for all the metadata


In [9]:


user_id = torch.tensor(data['user_id'].values, dtype = torch.long)
age_lookup = torch.tensor(user_details['age'].values, dtype = torch.float32)
gender_lookup = torch.tensor(user_details['gender'].values, dtype = torch.long)
occupation_lookup = torch.tensor(user_details['occupation'].values, dtype = torch.long)

movie_id = torch.tensor(data['movie_id'].values, dtype = torch.long)
genre_columns = [f"genre_{i}" for i in range(19)]
genre_lookup = torch.tensor(movie_details[genre_columns].values, dtype = torch.float32)

ratings = torch.tensor(data['rating'].values, dtype = torch.float32)
print(genre_columns)

print(len(user_id))
print(len(age_lookup))
print(len(ratings))

['genre_0', 'genre_1', 'genre_2', 'genre_3', 'genre_4', 'genre_5', 'genre_6', 'genre_7', 'genre_8', 'genre_9', 'genre_10', 'genre_11', 'genre_12', 'genre_13', 'genre_14', 'genre_15', 'genre_16', 'genre_17', 'genre_18']
100000
943
100000


# Splitting the data between train and test

In [10]:
train_user_id, test_user_id, train_movie_id, test_movie_id, train_ratings, test_ratings  = train_test_split(user_id, movie_id, ratings, test_size = .2, shuffle = True, random_state = 42)


# Creating a custom dataset to fetch the data columns as needed & also creating dataloaders


In [11]:
class MovieLensDataset(Dataset):

    def __init__(
        self,
        user_id,
        age_lookup,
        gender_lookup,
        occupation_lookup,
        movie_id,
        genre_lookup,
        ratings,
    ):

        self.ratings = ratings

        # ---------- User tensors ----------

        self.user_id = user_id

        self.user_age = age_lookup

        self.user_gender = gender_lookup

        self.user_occupation = occupation_lookup

        # ---------- Movie tensor ----------

        self.movie_id = movie_id

        self.movie_genres = genre_lookup

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        user_id = self.user_id[idx]
        movie_id = self.movie_id[idx]

        age = self.user_age[user_id]
        gender = self.user_gender[user_id]
        occupation = self.user_occupation[user_id]

        genres = self.movie_genres[movie_id]

        ratings = self.ratings[idx]
        return (
            user_id,
            age,
            gender,
            occupation,
            movie_id,
            genres,
            ratings
        )


train_dataset = MovieLensDataset(train_user_id, age_lookup, gender_lookup, occupation_lookup, train_movie_id, genre_lookup, train_ratings)
test_dataset = MovieLensDataset(test_user_id, age_lookup, gender_lookup, occupation_lookup, test_movie_id, genre_lookup, test_ratings)

trainloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
testloader = DataLoader(test_dataset, batch_size = len(test_dataset))


user_id_sample, age_sample, gender_sample, occupation_sample, movie_id_sample, genres_sample, ratings_sample = next(iter(trainloader))

print(user_id_sample.shape)
print('\n', age_sample.shape)
print('\n', gender_sample.shape)
print('\n', occupation_sample.shape)
print('\n', movie_id_sample[:5])
print('\n', genres_sample[:5])
print('\n', ratings_sample[:5])

torch.Size([128])

 torch.Size([128])

 torch.Size([128])

 torch.Size([128])

 tensor([247, 300,  14, 173, 122])

 tensor([[0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.],
        [0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0.]])

 tensor([3., 3., 4., 4., 3.])


In [12]:
num_movies = len(pd.unique(data['movie_id']))
num_users = len(pd.unique(data['user_id']))

# Model Class Creation

In [13]:
class TwoTowerModel(nn.Module):
    def __init__(self, num_user, num_movies, num_embeddings):
        super().__init__()

        self.user_embeddings = nn.Embedding(
            num_user,
            embedding_dim = num_embeddings
        )

        self.movie_embeddings = nn.Embedding(
            num_movies,
            embedding_dim = num_embeddings
        )

        self.gender_embeddings = nn.Embedding(
            2,
            embedding_dim = 4
        )

        self.occupation_embeddings = nn.Embedding(
            21,
            embedding_dim = 8
        )


        # self.user_bias = nn.Embedding(
        #     num_user,
        #     1
        # )

        # self.movie_bias = nn.Embedding(
        #     num_movies,
        #     1
        # )

        nunits_input_user = (
            self.occupation_embeddings.embedding_dim
            + self.gender_embeddings.embedding_dim
            + self.user_embeddings.embedding_dim
            + 1
        )

        self.num_genre = 19
        nunits_input_movie = (
            self.num_genre
            + self.movie_embeddings.embedding_dim
        )

        self.user_tower = nn.Sequential(
            nn.Linear(nunits_input_user, 64),
            nn.ReLU(),
            nn.Linear(64, num_embeddings)
        )

        self.movie_tower = nn.Sequential(
            nn.Linear(nunits_input_movie, 64),
            nn.ReLU(),
            nn.Linear(64, num_embeddings)
        )

    # User Tower
    def encode_user(self, user_id, age, gender, occupation):
        user_vec = self.user_embeddings(user_id)
        age_vec = age.unsqueeze(1)
        gender_vec = self.gender_embeddings(gender)
        occupation_vec = self.occupation_embeddings(occupation)
        user_features = torch.cat(
            [
                user_vec,
                gender_vec,
                occupation_vec,
                age_vec
            ],
            dim = 1
        )

        user_vec = self.user_tower(user_features)

        return user_vec


    # Movie Tower
    def encode_movie(self, movie_id, genre):

        movie_vec = self.movie_embeddings(movie_id)
        movie_features = torch.cat(
            [
                movie_vec,
                genre
            ],
            dim = 1
        )
        movie_vec = self.movie_tower(movie_features)

        return movie_vec

    # user_bias = self.user_bias(user_id)
    # movie_bias = self.movie_bias(movie_id)

    # user_bias = user_bias.squeeze()
    # movie_bias = movie_bias.squeeze()

    def forward(self,user_id,age,gender,occupation,movie_id,genres):

        user_vec = self.encode_user(
            user_id,
            age,
            gender,
            occupation
        )

        movie_vec = self.encode_movie(
            movie_id,
            genres
        )

        prediction = (user_vec * movie_vec).sum(dim = 1)

        return prediction

torch.manual_seed(42)

model = TwoTowerModel(943, 1682, 32)

lossfun = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr = 0.0001)

# Retrieving the Movie Data

In [14]:
def build_movie_index(model, movie_genre):
    model.eval()

    with torch.no_grad():
        movie_genre = movie_genre.to(device)
        movie_ids = torch.arange(len(movie_genre), device = device)
        movie_vectors = model.encode_movie(movie_ids, movie_genre)

    return movie_vectors

check1 = build_movie_index(model, genre_lookup)
check1.shape

torch.Size([1682, 32])

In [15]:
# Debugging
# print(age_lookup[5].shape)
# print(gender_lookup[5].shape)
# print(occupation_lookup[5].shape)
# print(torch.tensor(5).shape)

# print(age_lookup[5].shape)
# print(torch.tensor([age_lookup[5].shape]))
# dummy = age_lookup[5].unsqueeze(0)
# print(dummy.shape)


# Helper function to get user embedding


In [24]:
def get_userEmbeddings(model, user_id):
    model.eval()

    with torch.no_grad():
        age = torch.tensor([age_lookup[user_id]], device = device)
        gender = gender_lookup[user_id].unsqueeze(0).to(device)
        occupation = occupation_lookup[user_id].unsqueeze(0).to(device)
        user_id = torch.tensor([user_id], device = device)

        # Debugging checks
        # print(age.shape)
        # print(gender.shape)
        # print(user_id.shape)
        # print(occupation.shape)

        user_vector = model.encode_user(user_id, age, gender, occupation)

    return user_vector

check2 = get_userEmbeddings(model, 5)

check2.shape

torch.Size([1, 32])

# Training the Model

In [17]:
epochs = 30
trainloss = torch.zeros(epochs)
testloss = torch.zeros(epochs)
trainRmse = torch.zeros(epochs)
testRmse = torch.zeros(epochs)

model.to(device)

for epochi in range(epochs):

    model.train()

    batchLoss = []
    for user_id, age, gender, occupation, movie_id, genres, ratings in trainloader:


        user_id = user_id.to(device)
        age = age.to(device)
        gender = gender.to(device)
        occupation = occupation.to(device)

        movie_id = movie_id.to(device)
        genres = genres.to(device)
        ratings = ratings.to(device)

        prediction = model(
            user_id,
            age,
            gender,
            occupation,
            movie_id,
            genres
        )

        loss = lossfun(prediction, ratings)
        batchLoss.append(loss.detach())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    trainloss[epochi] = torch.mean(torch.stack(batchLoss))
    trainRmse[epochi] = torch.sqrt(trainloss[epochi])


    # Using test data to test the model
    # Tuserid, Tmovieid, Tratings = next(iter(testloader))
    # Tuserid, Tmovieid, Tratings = Tuserid.to(device), Tmovieid.to(device), Tratings.to(device)

    model.eval()

    with torch.no_grad():

        for Tuser_id, Tage, Tgender, Toccupation, Tmovie_id, Tgenres, Tratings in testloader:

            Tuser_id = Tuser_id.to(device)
            Tage = Tage.to(device)
            Tgender = Tgender.to(device)
            Toccupation = Toccupation.to(device)
            Tmovie_id = Tmovie_id.to(device)
            Tgenres = Tgenres.to(device)
            Tratings = Tratings.to(device)


            Tprediction = model(
                Tuser_id,
                Tage,
                Tgender,
                Toccupation,
                Tmovie_id,
                Tgenres
            )

            loss = lossfun(Tprediction,Tratings)
    testloss[epochi] = loss.item()
    testRmse[epochi] = torch.sqrt(loss)


    if epochi == epochs - 1:
        print(f"Final Results -> Train RMSE:{trainRmse[epochi]:.4f} & Test RMSE:{testRmse[epochi]:.4f}")
        print(f"Final Results -> Train loss:{trainloss[epochi]:.4f} & Test loss:{loss.item():.4f}", end = '\n\n')


Final Results -> Train RMSE:0.9137 & Test RMSE:0.9545
Final Results -> Train loss:0.8349 & Test loss:0.9111



In [18]:
first_15 = np.round(Tprediction.cpu().numpy()[:15], decimals = 0)
print(first_15)
print(Tratings.cpu().numpy()[:15])

print(Tprediction.max())
print(Tprediction.min())

[4. 4. 4. 3. 4. 3. 3. 4. 3. 3. 3. 4. 3. 4. 4.]
[4. 3. 4. 2. 2. 3. 5. 4. 3. 4. 4. 4. 3. 4. 3.]
tensor(5.9629)
tensor(0.5786)


In [42]:
vec_movie = build_movie_index(model, genre_lookup)

In [49]:
def recommend_movies(model, user_id, top_k = 10):
    user_emb = get_userEmbeddings(model, user_id)
    scores = vec_movie @ user_emb.T

    # print(scores.shape)
    # print(scores.min(), '\n', scores.max(), '\n', scores.mean())

    scores.squeeze_()
    watched_movies = data[data["user_id"] == user_id]["movie_id"].values
    scores[watched_movies] = -float('inf')

    # print(scores[watched_movies])

    values, idx = torch.topk(scores, k = top_k)

    return idx, values

user10_idx, user10_values = recommend_movies(model, 10)

print(f"The top 10 movie recommendations for user 10 are as follows: \nMovie_ID: {user10_idx}\n&\nThe score is {user10_values}")

The top 10 movie recommendations for user 10 are as follows: 
Movie_ID: tensor([1366, 1234, 1656, 1525, 1428, 1155, 1636, 1422,  512, 1461])
&
The score is tensor([4.6838, 4.6587, 4.5748, 4.5727, 4.4684, 4.4507, 4.4234, 4.4206, 4.4004,
        4.3452])
